### Cell 1 (Markdown): What this notebook does

Runs the GPU smoke test (ops/gpu_smoke.py) for one Kaggle candidate from
configs/gpu_candidates.yaml. The test checks the GPUs, the NVIDIA driver and the free disk,
downloads the pinned server and model files with SHA-256 checks, starts the server, asks five
fixed public questions and writes one result bundle with the answers and the measured times.

Before running:

- Settings, Accelerator: GPU T4 x2.
- Settings, Internet: on (the code, the packages and the models are downloaded).
- Cell 2: set COMMIT (the full 40-character ID of the commit to test) and CANDIDATE.

Run the cells in order, one candidate per run. Cell 10 frees the disk before the next candidate.
Large files go to /kaggle/tmp/mrag and are not kept after the session. The result bundle goes to /kaggle/working/bundles, which is the notebook output.
Public files only, so no token is needed.

In [ ]:
# Cell 2 (Code): Settings and helpers
import json
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

# The full 40-character ID of the commit to test
COMMIT = ""
# The Kaggle candidates in configs/gpu_candidates.yaml. Choose one per run
CANDIDATES = ["kaggle-llamacpp-27b-4bit", "kaggle-llamacpp-9b-16bit", "kaggle-vllm-9b-16bit"]
CANDIDATE = "kaggle-llamacpp-9b-16bit"

REPO_URL = "https://github.com/mdu-ai-lab/multiagent-ai-lab.git"
# The same uv version as CI
UV_VERSION = "0.13.7"
# Large files: the code, the environments and the models. They are not kept
WORK = Path("/kaggle/tmp/mrag")
REPO = WORK / "repo"
SMOKE = WORK / "smoke"
# Only the small result bundles go to the notebook output
OUTPUT = Path("/kaggle/working/bundles")
# The uv cache sits next to the environments, so files are linked instead of copied
os.environ["UV_CACHE_DIR"] = str(WORK / "uv-cache")

if not re.fullmatch(r"[0-9a-f]{40}", COMMIT):
    raise ValueError("Set COMMIT to the full 40-character ID of the commit to test")
if CANDIDATE not in CANDIDATES:
    raise ValueError(f"CANDIDATE must be one of {CANDIDATES}")


def run(command, cwd=None, env=None):
    # Shows the command and stops the notebook when it fails. A shell line that starts
    # with ! would not stop the notebook on a failure
    print("$", " ".join(map(str, command)), flush=True)
    subprocess.run([str(part) for part in command], cwd=cwd, env=env, check=True)


def output(command, cwd=None):
    result = subprocess.run(
        [str(part) for part in command], cwd=cwd, check=True, capture_output=True, text=True
    )
    return result.stdout.strip()

In [ ]:
# Cell 3 (Code): Check the machine (GPUs, driver, disk and memory)
run(["nvidia-smi"])
WORK.mkdir(parents=True, exist_ok=True)
for folder in (WORK, Path("/kaggle/working")):
    usage = shutil.disk_usage(folder)
    print(f"Disk at {folder}: {usage.free / 1e9:.1f} GB free of {usage.total / 1e9:.1f} GB")
memory = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")
print(f"Memory: {memory / 1e9:.1f} GB")
print("Python of the notebook:", sys.version.split()[0])

In [ ]:
# Cell 4 (Code): Install uv
run([sys.executable, "-m", "pip", "install", "--quiet", f"uv=={UV_VERSION}"])
UV = output([sys.executable, "-c", "import uv; print(uv.find_uv_bin())"])
version = output([UV, "--version"])
print(version)
if version.split()[:2] != ["uv", UV_VERSION]:
    raise RuntimeError(f"uv {UV_VERSION} was expected")

In [ ]:
# Cell 5 (Code): Get the code at the chosen commit
if not REPO.exists():
    run(["git", "clone", "--quiet", REPO_URL, REPO])
run(["git", "-C", REPO, "fetch", "--quiet", "origin", COMMIT])
run(["git", "-C", REPO, "checkout", "--quiet", "--detach", COMMIT])
if output(["git", "-C", REPO, "rev-parse", "HEAD"]) != COMMIT:
    raise RuntimeError("The code folder is not at COMMIT")
if output(["git", "-C", REPO, "status", "--porcelain"]):
    raise RuntimeError(f"The code in {REPO} has changes. Delete the folder and run this cell again")
print("Code at commit", COMMIT)

In [ ]:
# Cell 6 (Code): Install the platform environment from uv.lock
run([UV, "sync", "--frozen", "--no-dev"], cwd=REPO)
PYTHON = REPO / ".venv" / "bin" / "python"
# The server of the candidate, read from configs/gpu_candidates.yaml at this commit
READ_SERVER = (
    "import sys, yaml\n"
    "candidates = yaml.safe_load(open('configs/gpu_candidates.yaml'))['candidates']\n"
    "print(candidates[sys.argv[1]]['server'])"
)
SERVER = output([PYTHON, "-c", READ_SERVER, CANDIDATE], cwd=REPO)
print(f"{CANDIDATE} runs on {SERVER}")

In [ ]:
# Cell 7 (Code): Install vLLM from its lock in ops/vllm-server (vLLM candidates only)
VLLM = REPO / "ops" / "vllm-server" / ".venv" / "bin" / "vllm"
if SERVER == "vllm":
    run([UV, "sync", "--frozen"], cwd=REPO / "ops" / "vllm-server")
    print("vLLM", output([VLLM, "--version"]))
else:
    print(f"{CANDIDATE} uses llama.cpp, which needs no Python packages")

In [ ]:
# Cell 8 (Code): Run the smoke test
command = [PYTHON, "ops/gpu_smoke.py", "--candidate", CANDIDATE, "--work-dir", SMOKE]
if SERVER == "vllm":
    command += ["--vllm", VLLM]
# uv on the PATH, so the smoke test can record the installed vLLM packages
env = os.environ | {"PATH": f"{Path(UV).parent}{os.pathsep}{os.environ['PATH']}"}
run(command, cwd=REPO, env=env)

In [ ]:
# Cell 9 (Code): Copy the result bundles to the notebook output and show the answers
OUTPUT.mkdir(parents=True, exist_ok=True)
for bundle in sorted((SMOKE / "bundles").iterdir()):
    shutil.copytree(bundle, OUTPUT / bundle.name, dirs_exist_ok=True)
    print(bundle.name, json.loads((bundle / "bundle.json").read_text())["metrics"])
for line in (SMOKE / f"answers_{CANDIDATE}.jsonl").read_text().splitlines():
    record = json.loads(line)
    print("correct" if record["correct"] else "WRONG", record["question"], repr(record["answer"]))

In [ ]:
# Cell 10 (Code): Free the disk before the next candidate
# Deletes the downloaded models only. The code, the environments, the llama.cpp build and
# the results stay
shutil.rmtree(SMOKE / "models", ignore_errors=True)
print(f"Models deleted. Disk at {WORK}: {shutil.disk_usage(WORK).free / 1e9:.1f} GB free")

### Cell 11 (Markdown): After the run, on the Mac

1. Download the bundles folder from the notebook output. Put each bundle folder into
   data/inbox/kaggle/ in the project folder.
2. Start the MLflow server in one terminal: make mlflow
3. In a second terminal, in the project folder: make pull-inbox

make pull-inbox adds each bundle once to the Mac's MLflow. A bundle that is already there is
skipped.